In [1]:
from pydantic import BaseModel, Field
from typing import List
from langchain_groq import ChatGroq

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from tqdm import tqdm

model_name_openai = "gpt-4o-2024-08-06" # Garante 100% do output em JSON

llm_openai = ChatOpenAI(
    model=model_name_openai, # 100% json output
    temperature=0,
)

system_prompt = """
Você é um assistente inteligente que vai construir uma conversa entre duas personas. Considere que a conversa pode ser natural (pode ser como num formato de podcast). Seu trabalho e gerar uma conversa extensa que começa com tópicos relacionados à {topic_1}, mas que gradualmente se move para tópicos relacionados a {topic_2}.

Siga o seguinte padrão para a conversa:

Persona 1: [fala algo que pode ser relacionado a {topic_1}]
Persona 2: [responde, ainda relacionado a {topic_1} ou fazendo uma transição suave]
... 


---------------

Continue a conversa por pelo menos 100 trocas de mensagens
"""

prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "Gere a conversa entre as duas personas."),
        ]
)

chain = prompt | llm_openai | StrOutputParser()

In [2]:
response = chain.invoke({
    "topic_1": "história",
    "topic_2": "educação financeira"
})

In [3]:
def select_persona(message: str) -> str:
    if message.startswith("Persona 1:"):
        return "Persona 1"
    elif message.startswith("Persona 2:"):
        return "Persona 2"
    else:
        return "Unknown"

conversation_list = response.split("\n")
conversation_list = [line for line in conversation_list if line.strip() != ""]
conversation_list_persona = [{"persona": select_persona(line), "message": line[11:]} for line in conversation_list]

In [4]:
system_prompt_summary = """
Você é um assistente inteligente que vai resumir uma conversa entre duas personas. Você vai receber uma lista de mensagens, cada uma associada a uma persona (Persona 1 ou Persona 2). Seu trabalho é gerar um resumo conciso da conversa, dê enfase para o tópico principal discutido na conversa no momento atual (mesmo que o tópico tenha mudado ao longo da conversa). Os tópicos devem girar em torno de {topic_1} e {topic_2}.
"""

prompt_summary = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt_summary),
            ("human", "Resuma a conversa dada: {conversation}"),
        ]
)

chain_summary = prompt_summary | llm_openai | StrOutputParser()

In [11]:
summary_list = []
buffer = []
for i in tqdm(range(len(conversation_list))):
    buffer.append(conversation_list[i])
    buffer_trimmed = buffer[-10:]  # keep only the last 10 messages
    buffer_str = "\n".join(buffer_trimmed)
    response_summary = chain_summary.invoke({
        "topic_1": "história",
        "topic_2": "educação financeira",
        "conversation": buffer_str
    })
    summary_list.append(response_summary)

100%|██████████| 58/58 [02:42<00:00,  2.80s/it]


In [12]:
len(conversation_list), len(summary_list), len(conversation_list_persona)

(58, 58, 58)

In [17]:
all_data_list = []
for i in range(len(conversation_list)):
    all_data_list.append({
        "summary": summary_list[i],
        "persona": conversation_list_persona[i]["persona"],
        "message": conversation_list_persona[i]["message"]
    })

In [18]:
# save list of dicts to json file as a list of json objects
import json

with open("../data/conversation/conversation_data_2.json", "w", encoding="utf-8") as f:
    json.dump(all_data_list, f, ensure_ascii=False, indent=4)

In [19]:
# Load data from json file

with open("../data/conversation/conversation_data_2.json", "r", encoding="utf-8") as f:
    loaded_content = json.load(f)   
    print(loaded_content[:5])

[{'summary': 'Persona 1 expressa fascínio pela história da humanidade, destacando a Revolução Industrial como um ponto de virada significativo.', 'persona': 'Persona 1', 'message': 'Você já parou para pensar em como a história da humanidade é fascinante? Por exemplo, a Revolução Industrial foi um ponto de virada incrível, não acha?'}, {'summary': 'A conversa gira em torno da história da humanidade, com foco na Revolução Industrial. Persona 1 destaca a importância desse evento como um ponto de virada fascinante, enquanto Persona 2 concorda, enfatizando o impacto profundo que a Revolução Industrial teve na produção, sociedade e economia, moldando o mundo moderno.', 'persona': 'Persona 2', 'message': 'Com certeza! A Revolução Industrial não só transformou a maneira como produzimos bens, mas também teve um impacto profundo na sociedade e na economia. É interessante como essas mudanças moldaram o mundo moderno.'}, {'summary': 'A conversa entre as duas personas gira em torno da história da h

In [20]:
loaded_content[:5]

[{'summary': 'Persona 1 expressa fascínio pela história da humanidade, destacando a Revolução Industrial como um ponto de virada significativo.',
  'persona': 'Persona 1',
  'message': 'Você já parou para pensar em como a história da humanidade é fascinante? Por exemplo, a Revolução Industrial foi um ponto de virada incrível, não acha?'},
 {'summary': 'A conversa gira em torno da história da humanidade, com foco na Revolução Industrial. Persona 1 destaca a importância desse evento como um ponto de virada fascinante, enquanto Persona 2 concorda, enfatizando o impacto profundo que a Revolução Industrial teve na produção, sociedade e economia, moldando o mundo moderno.',
  'persona': 'Persona 2',
  'message': 'Com certeza! A Revolução Industrial não só transformou a maneira como produzimos bens, mas também teve um impacto profundo na sociedade e na economia. É interessante como essas mudanças moldaram o mundo moderno.'},
 {'summary': 'A conversa entre as duas personas gira em torno da his